## Databricks / R Data Pipeline

This Databricks notebook provides an example of a solution to migrating a process that uses both SQL and R. This deomnstrates the 'Move all code into Databricks' solution detailed in the workbook. _Currently this only involves running a notebook, however it could also be incorporated into a job - maybe update to add before finalising_

Pipeline stages:
1. **Check data** - Look at data structure of Autumn 2025 pupils and schools data
2. **Data Manipulation** - Select columns and parse metadata, join pupils to schools
3. **Outputs** - Produce average pupil premium by Ofsted rating summary

In [0]:
-- Define catalog being used
USE CATALOG catalog_40_copper_analyst_training;

### Check data
Look at data structure.

In [0]:
-- Check structure of pupils_autumn_2025 table
SELECT * FROM catalog_40_copper_analyst_training.bronze.pupils_autumn_2025 LIMIT 1000

In [0]:
-- Check structure of schools_autumn_2025 table
SELECT * FROM catalog_40_copper_analyst_training.bronze.schools_autumn_2025 LIMIT 1000

### Data Manipulation
Select required columns and parse JSON metadata, join pupils to schools.

In [0]:
-- Pupils table: select columns and parse JSON metadata
CREATE OR REPLACE TEMPORARY VIEW pupils_aut AS
SELECT 
  pupil_id,
  gender,
  school_urn,
  metadata_json:address:postcode AS postcode,
  LOWER(metadata_json:sen_status) AS sen_status,
  metadata_json:fsm_eligible AS fsm_status
FROM catalog_40_copper_analyst_training.bronze.pupils_autumn_2025;

-- Schools table: select columns, parse JSON, reformat data in city column, deduplicate
CREATE OR REPLACE TEMP VIEW schools_aut AS -- We use TEMP here instead of TEMPORARY in the above example - these can be used interchangeably
SELECT DISTINCT
  school_urn,
  INITCAP(city) AS city,
  school_type,
  REPLACE(metadata_json:ofsted_rating, '"', '') AS ofsted_rating,
  CAST(metadata_json:pupil_premium_pct AS DOUBLE) AS pupil_premium_pct
FROM catalog_40_copper_analyst_training.bronze.schools_autumn_2025;

-- Join pupils and schools tables ahead of performing calculations
CREATE OR REPLACE TEMP TABLE pupils_schools_aut AS -- and here we create a temporary table rather than a view
SELECT 
  p.*,
  s.city,
  s.school_type,
  s.ofsted_rating,
  s.pupil_premium_pct
FROM pupils_aut p
LEFT JOIN schools_aut s
  ON p.school_urn = s.school_urn

### Outputs
Here we switch to using R to produce summary outputs. 

Need to figure out how to actually get this to work...

In [0]:
# Investigate relationships in joined data ----
cat(
  "Joined pupils_schools_aut dimensions: ",
  nrow(pupils_schools_aut),
  "x",
  ncol(pupils_schools_aut),
  "\n"
)

# Gender by Ofsted rating:
print(pupils_schools_aut %>% count(gender, ofsted_rating, sort = TRUE))

# SEN status by Ofsted rating:
print(pupils_schools_aut %>% count(sen_status, ofsted_rating, sort = TRUE))

# FSM status by Ofsted rating:
print(pupils_schools_aut %>% count(fsm_status, ofsted_rating, sort = TRUE))

# Average pupil premium percentage by Ofsted rating:
print(
  pupils_schools_aut %>%
    group_by(ofsted_rating) %>%
    summarise(
      pupils = n(),
      avg_pupil_premium_pct = mean(pupil_premium_pct, na.rm = TRUE),
      median_pupil_premium_pct = median(pupil_premium_pct, na.rm = TRUE),
      .groups = "drop"
    ) %>%
    arrange(desc(pupils))
)

# School count by city and Ofsted rating:
print(
  pupils_schools_aut %>%
    distinct(school_urn, city, ofsted_rating) %>%
    count(city, ofsted_rating, sort = TRUE)
)

# save pipeline outputs into the outputs folder
avg_pupil_premium_by_ofsted <- pupils_schools_aut %>%
  group_by(ofsted_rating) %>%
  summarise(
    pupils = n(),
    avg_pupil_premium_pct = mean(pupil_premium_pct, na.rm = TRUE),
    median_pupil_premium_pct = median(pupil_premium_pct, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(pupils))

write.csv(
  avg_pupil_premium_by_ofsted,
  file = "Databricks_workshops/Databricks workshop/03 Migrating existing processes/sql_and_r_combination/outputs/avg_pupil_premium_by_ofsted.csv",
  row.names = FALSE
)
